# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/
MemTotal: 1007.72 GB
MemFree: 283.79 GB
MemAvailable: 697.99 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successf

## 2. Llama-3-8B

In [3]:
import torch
from transformers import AutoModelForCausalLM
from collections import defaultdict
import numpy as np

def analyze_model_dtypes(model_name: str):
    # Load model
    print(f"Loading model {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    
    # Initialize counters
    dtype_counts = defaultdict(int)
    total_params = 0
    
    # Analyze parameters
    print("\nAnalyzing parameter dtypes...")
    for name, param in model.named_parameters():
        num_params = param.numel()
        dtype_counts[param.dtype] += num_params
        total_params += num_params
        
        # Print details for each layer
        print(f"{name}: {param.dtype} (shape: {param.shape})")
    
    # Calculate and print statistics
    print("\nSummary:")
    print(f"Total parameters: {total_params:,}")
    for dtype, count in dtype_counts.items():
        percentage = (count / total_params) * 100
        print(f"{dtype}: {count:,} parameters ({percentage:.2f}%)")

if __name__ == "__main__":
    model_name = "meta-llama/Meta-Llama-3-8B"
    analyze_model_dtypes(model_name)

Loading model meta-llama/Meta-Llama-3-8B...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing parameter dtypes...
model.embed_tokens.weight: torch.bfloat16 (shape: torch.Size([128256, 4096]))
model.layers.0.self_attn.q_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.0.self_attn.k_proj.weight: torch.bfloat16 (shape: torch.Size([1024, 4096]))
model.layers.0.self_attn.v_proj.weight: torch.bfloat16 (shape: torch.Size([1024, 4096]))
model.layers.0.self_attn.o_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.0.mlp.gate_proj.weight: torch.bfloat16 (shape: torch.Size([14336, 4096]))
model.layers.0.mlp.up_proj.weight: torch.bfloat16 (shape: torch.Size([14336, 4096]))
model.layers.0.mlp.down_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 14336]))
model.layers.0.input_layernorm.weight: torch.bfloat16 (shape: torch.Size([4096]))
model.layers.0.post_attention_layernorm.weight: torch.bfloat16 (shape: torch.Size([4096]))
model.layers.1.self_attn.q_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.1

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


## 3. All models

In [3]:
import torch
from collections import defaultdict
import logging
from typing import List, Optional
import os

from transformers import AutoModelForCausalLM
from hqq.engine.hf import HQQModelForCausalLM
from hqq.models.hf.base import AutoHQQHFModel
from awq import AutoAWQForCausalLM

from src.models import (
    base_models,
    hf_quantized_models,
    local_quantized_models,
    local_tokenizers
)

logger = logging.getLogger("dtype_analyzer")

def analyze_model_dtypes(
    model_name: str,
    include_layer_details: bool = False,
    cache_dir: Optional[str] = None
) -> dict:
    """
    Analyze the data types of model parameters.
    
    Args:
        model_name: Name of the model to analyze
        include_layer_details: Whether to print details for each layer
        cache_dir: Optional cache directory for model loading
    
    Returns:
        dict: Summary statistics of parameter dtypes
    """
    try:
        logger.info(f"Loading model: {model_name}")
        
        # Determine model path and type
        if model_name in local_quantized_models:
            model_path = local_quantized_models[model_name]
            is_local_model = True
        elif model_name in hf_quantized_models:
            model_path = hf_quantized_models[model_name]
            is_local_model = False
        elif model_name in base_models:
            model_path = base_models[model_name]
            is_local_model = False
        else:
            raise ValueError(f"Model {model_name} not found in model dictionaries")

        # Load appropriate model type
        if "HQQ" in model_name:
            try:
                model = HQQModelForCausalLM.from_quantized(model_path, device_map='auto', cache_dir=cache_dir)
            except:
                model = AutoHQQHFModel.from_quantized(model_path, device_map='auto', cache_dir=cache_dir)
        elif "AWQ" in model_name:
            model = AutoAWQForCausalLM.from_pretrained(
                model_path,
                trust_remote_code=True,
                torch_dtype=torch.float16,
                device_map='auto',
                cache_dir=cache_dir
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                torch_dtype="auto",
                device_map="auto",
                cache_dir=cache_dir
            )

        # Initialize counters
        dtype_counts = defaultdict(int)
        total_params = 0

        # Analyze parameters
        for name, param in model.named_parameters():
            num_params = param.numel()
            dtype_counts[param.dtype] += num_params
            total_params += num_params
            
            if include_layer_details:
                logger.info(f"{name}: {param.dtype} (shape: {param.shape})")

        # Create summary dictionary
        summary = {
            "model_name": model_name,
            "total_params": total_params,
            "dtype_distribution": {
                str(dtype): {
                    "count": count,
                    "percentage": (count / total_params) * 100
                }
                for dtype, count in dtype_counts.items()
            }
        }

        # Clean up
        del model
        torch.cuda.empty_cache()
        
        return summary

    except Exception as e:
        logger.error(f"Error analyzing {model_name}: {str(e)}")
        return {
            "model_name": model_name,
            "error": str(e)
        }

def analyze_multiple_models(
    analyze_base: bool = True,
    analyze_hf: bool = True,
    analyze_local: bool = True,
    include_layer_details: bool = False,
    cache_dir: Optional[str] = None
):
    """
    Analyze multiple models based on specified categories.
    """
    models_to_analyze = []
    
    if analyze_base:
        models_to_analyze.extend(base_models.keys())
    if analyze_hf:
        models_to_analyze.extend(hf_quantized_models.keys())
    if analyze_local:
        models_to_analyze.extend(local_quantized_models.keys())

    print(f"\nAnalyzing {len(models_to_analyze)} models...")
    
    results = []
    for model_name in models_to_analyze:
        summary = analyze_model_dtypes(model_name, include_layer_details, cache_dir)
        
        if "error" in summary:
            print(f"\n{model_name} analysis failed: {summary['error']}")
            continue
            
        print(f"\nSummary for {model_name}:")
        print(f"Total parameters: {summary['total_params']:,}")
        for dtype, info in summary['dtype_distribution'].items():
            print(f"{dtype}: {info['count']:,} parameters ({info['percentage']:.2f}%)")
        
        results.append(summary)
    
    return results

if __name__ == "__main__":
    # Configure logging
    logging.basicConfig(level=logging.INFO)
    
    # Example usage
    results = analyze_multiple_models(
        analyze_base=True,    # Set to False to skip base models
        analyze_hf=True,      # Set to False to skip HF quantized models
        analyze_local=True,   # Set to False to skip local quantized models
        include_layer_details=False,  # Set to True to see per-layer details
        cache_dir=None  # Specify cache directory if needed
    )

2024-10-27 10:06:27.944280: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-27 10:06:27.965600: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-27 10:06:27.971333: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-27 10:06:27.986285: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-27 10:06:30.249401: W tensorflow/compiler/tf2


Analyzing 23 models...


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
INFO:dtype_analyzer:Loading model: TinyLlama



Summary for TinyLlama-Chat:
Total parameters: 1,100,048,384
torch.bfloat16: 1,100,048,384 parameters (100.00%)


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
INFO:dtype_analyzer:Loading model: Llama-3-8B
INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


...


Summary for Llama-3-8B-HQQ-8-uniform-local:
Total parameters: 8,030,261,248
torch.float16: 1,050,939,392 parameters (13.09%)
torch.uint8: 6,979,321,856 parameters (86.91%)


100%|██████████| 225/225 [00:00<00:00, 10357.56it/s]
INFO:dtype_analyzer:Loading model: Llama-3-8B-QUANTO-local
ERROR:dtype_analyzer:Error analyzing Llama-3-8B-QUANTO-local: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO'. Please provide either the path to a local folder or the repo_id of a model on the Hub.
INFO:dtype_analyzer:Loading model: Llama-3-8B-QUANTO-CALIB-local
ERROR:dtype_analyzer:Error analyzing Llama-3-8B-QUANTO-CALIB-local: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-CALIB'. Please provide either the path to a local folder or the repo_id of a model on the Hub.
INFO:dtype_analyzer:Loading model: Llama-3-8B-QUANTO-QAT-local
ERROR:dtype_analyzer:Error analyzing Llama-3-8B-QUANTO-QAT-local: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-QAT'. Please provide either the path to a local folder or the repo_id of a model on the Hub.
INFO:dtype_analyzer:Loading model: Llama-3-8B-HQQ-LOR


Summary for Llama-3-8B-HQQ-mixed-local:
Total parameters: 2,426,671,104
torch.float16: 1,050,939,392 parameters (43.31%)
torch.uint8: 671,088,640 parameters (27.65%)
torch.int32: 704,643,072 parameters (29.04%)

Llama-3-8B-QUANTO-local analysis failed: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

Llama-3-8B-QUANTO-CALIB-local analysis failed: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-CALIB'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

Llama-3-8B-QUANTO-QAT-local analysis failed: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-QAT'. Please provide either the path to a local folder or the repo_id of a model on the Hub.


  0%|          | 0/225 [00:00<?, ?it/s]
ERROR:dtype_analyzer:Error analyzing Llama-3-8B-HQQ-LORA-local: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.
INFO:dtype_analyzer:Loading model: Llama-3-8B-AQLM-LORA-local
ERROR:dtype_analyzer:Error analyzing Llama-3-8B-AQLM-LORA-local: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-AQLM-LORA'. Please provide either the path to a local folder or the repo_id of a model on the Hub.



Llama-3-8B-HQQ-LORA-local analysis failed: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

Llama-3-8B-AQLM-LORA-local analysis failed: Incorrect path_or_model_id: '/nfs/students/daro/models/Meta-Llama-3-8B-AQLM-LORA'. Please provide either the path to a local folder or the repo_id of a model on the Hub.
